# Module 05 — Neural Network Foundations
## 05-02: Multilayer Perceptrons (MLP)

**Objective:** Build a Multilayer Perceptron from scratch, understand the Universal
Approximation Theorem, and train a deep MLP on FashionMNIST.

**Prerequisites:** 05-01 (Perceptron & Linear Classifiers), 01-03 (Calculus),
01-02 (Linear Algebra)


---
## Part 0 — Setup & Prerequisites

This notebook covers the **Multilayer Perceptron (MLP)** — a stack of linear layers
separated by non-linear activations.  Adding hidden layers gives the model the ability
to represent *any* continuous function (Universal Approximation Theorem) and to solve
problems that a single Perceptron cannot, including XOR.

We will:

1. Implement the MLP forward pass from scratch using NumPy.
2. Demonstrate that a 2-layer MLP with the right weights solves XOR.
3. Visualise the Universal Approximation Theorem empirically.
4. Wrap the MLP into an `nn.Module` and verify numerical equivalence.
5. Train on **FashionMNIST** (10-class image classification).
6. Run a depth vs width ablation to understand architectural trade-offs.

**Note on activations:** We use ReLU here as a standard choice.
A detailed study of activation functions (ReLU, GELU, Swish, etc.) is in **05-03**.

**Note on backpropagation:** Training uses PyTorch autograd.
The manual derivation of backpropagation is in **05-06**.


In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import sys
import time
import warnings
import random
from typing import Any
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

import torchvision
import torchvision.transforms as transforms

from sklearn.datasets import make_moons, make_classification

from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.decomposition import PCA
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"NumPy: {np.__version__}")
if torch.cuda.is_available():
    print(f"CUDA: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")


In [ ]:
# ── Reproducibility & Device ──────────────────────────────────────────────────
SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
BATCH_SIZE        = 256
NUM_EPOCHS        = 15
LEARNING_RATE     = 1e-3
HIDDEN_DIMS       = [256, 128]   # Hidden layer sizes for the main MLP
INPUT_DIM         = 784          # 28 * 28 (FashionMNIST flattened)
NUM_CLASSES       = 10
NUM_EPOCHS_ABLAT  = 5            # Epochs for the depth-vs-width ablation
DATA_DIR          = "../data"

FASHION_CLASSES = [
    "T-shirt", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot",
]


In [ ]:
# ── Load FashionMNIST (official train / test splits) ─────────────────────────
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.2860,), (0.3530,)),   # FashionMNIST channel stats
])

full_train_set = torchvision.datasets.FashionMNIST(
    root=DATA_DIR, train=True, download=True, transform=transform,
)
test_set = torchvision.datasets.FashionMNIST(
    root=DATA_DIR, train=False, download=True, transform=transform,
)

# 90 / 10 split of the official training set
generator = torch.Generator().manual_seed(SEED)
train_size = int(0.9 * len(full_train_set))
val_size   = len(full_train_set) - train_size
train_set, val_set = torch.utils.data.random_split(
    full_train_set, [train_size, val_size], generator=generator,
)

train_loader = DataLoader(
    train_set, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=0, pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    val_set, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=0, pin_memory=torch.cuda.is_available(),
)
test_loader = DataLoader(
    test_set, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=0, pin_memory=torch.cuda.is_available(),
)

print(f"FashionMNIST  train={train_size}  val={val_size}  test={len(test_set)}")
print(f"Input shape: 1 x 28 x 28 → {INPUT_DIM} dims after flattening")
print(f"Classes ({NUM_CLASSES}): {FASHION_CLASSES}")

# ── Visualise sample images from each class ───────────────────────────────────
fig_eda, axes_eda = plt.subplots(2, 5, figsize=(14, 5))
axes_eda_flat = axes_eda.ravel()
# Get one sample per class from the raw (un-normalised) dataset
raw_train = torchvision.datasets.FashionMNIST(
    root=DATA_DIR, train=True, download=False, transform=transforms.ToTensor(),
)
for cls_idx in range(NUM_CLASSES):
    sample_idx = next(i for i, (_, lbl) in enumerate(raw_train) if lbl == cls_idx)
    img_raw, _ = raw_train[sample_idx]
    axes_eda_flat[cls_idx].imshow(img_raw.squeeze(), cmap="gray")
    axes_eda_flat[cls_idx].set_title(FASHION_CLASSES[cls_idx], fontsize=9)
    axes_eda_flat[cls_idx].axis("off")
plt.suptitle("FashionMNIST — One Sample per Class", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

# Class distribution
targets_all = [lbl for _, lbl in full_train_set]
class_counts = np.bincount(targets_all)
print("\nClass distribution (training set):")
for i, (cls, cnt) in enumerate(zip(FASHION_CLASSES, class_counts)):
    print(f"  {i}: {cls:<12s}  {cnt:6d} samples")


---
## Part 1 — MLP Theory & From-Scratch Implementation

### 1.1 Mathematical Definition

An $L$-layer MLP computes a function $f: \mathbb{R}^{d_0} \to \mathbb{R}^{d_L}$ via:

$$\mathbf{h}^{(0)} = \mathbf{x}, \qquad
\mathbf{h}^{(l)} = \sigma\!\left(\mathbf{W}^{(l)} \mathbf{h}^{(l-1)} + \mathbf{b}^{(l)}\right),
\quad l = 1, \ldots, L-1$$

$$\hat{\mathbf{y}} = \mathrm{softmax}\!\left(\mathbf{W}^{(L)} \mathbf{h}^{(L-1)} + \mathbf{b}^{(L)}\right)$$

where $\sigma$ is a non-linear activation function (we use **ReLU** here; see **05-03** for
a full study of activation functions).

### 1.2 Universal Approximation Theorem (Cybenko 1989, Hornik 1991)

> A feedforward network with a single hidden layer of sufficient width can approximate
> any continuous function on a compact subset of $\mathbb{R}^n$ to arbitrary precision.

This means: given enough neurons, **any** function can be approximated.  In practice,
**depth** (more layers) is more parameter-efficient than **width** (wider single layer) —
this motivates deep networks.

### 1.3 Forward Pass Implementation


In [ ]:
# ── 1.3 Forward Pass Building Blocks ─────────────────────────────────────────
def linear_layer(
    X: np.ndarray,
    W: np.ndarray,
    b: np.ndarray,
) -> np.ndarray:
    '''Compute a linear (affine) transformation: Z = X @ W + b.

    Args:
        X: Input of shape (n_samples, n_in).
        W: Weight matrix of shape (n_in, n_out).
        b: Bias vector of shape (n_out,).

    Returns:
        Pre-activation output of shape (n_samples, n_out).
    '''
    return X @ W + b


def relu(Z: np.ndarray) -> np.ndarray:
    '''Apply Rectified Linear Unit element-wise: ReLU(z) = max(0, z).

    This is a preview — activation functions are studied in depth in 05-03.

    Args:
        Z: Array of any shape containing pre-activation values.

    Returns:
        Array of same shape with negative values zeroed out.
    '''
    return np.maximum(0.0, Z)


def softmax(Z: np.ndarray) -> np.ndarray:
    '''Compute numerically stable softmax over the last axis.

    Uses the log-sum-exp trick: subtract max before exponentiation to
    prevent overflow when logits are large.

    Args:
        Z: Logit array of shape (n_samples, n_classes).

    Returns:
        Probability array of shape (n_samples, n_classes). Rows sum to 1.
    '''
    Z_shifted = Z - Z.max(axis=-1, keepdims=True)   # subtract max for stability
    exp_Z     = np.exp(Z_shifted)
    return exp_Z / exp_Z.sum(axis=-1, keepdims=True)


# ── Quick verification ─────────────────────────────────────────────────────────
rng_test  = np.random.default_rng(SEED)
X_test_fn = rng_test.standard_normal((4, 3))
W_test_fn = rng_test.standard_normal((3, 2))
b_test_fn = np.zeros(2)

Z_out  = linear_layer(X_test_fn, W_test_fn, b_test_fn)
H_out  = relu(Z_out)
P_out  = softmax(Z_out)

print("Building block verification:")
print(f"  Input X:       shape {X_test_fn.shape}")
print(f"  linear_layer:  shape {Z_out.shape},  values (first row): {Z_out[0].round(4)}")
print(f"  relu:          min={H_out.min():.4f}  (no negative values)")
print(f"  softmax:       rows sum to {P_out.sum(axis=1).round(6)}")

# Visualise relu vs sigmoid vs identity on a simple range
z_range = np.linspace(-3, 3, 300)
fig_fn, ax_fn = plt.subplots(figsize=(8, 3.5))
ax_fn.plot(z_range, relu(z_range[:, None]).ravel(),   lw=2, color="#2ca02c",
           label="ReLU (used here)")
ax_fn.plot(z_range, 1/(1+np.exp(-z_range)),           lw=2, color="#ff7f0e",
           ls="--", label="Sigmoid")
ax_fn.plot(z_range, z_range,                           lw=1.5, color="gray",
           ls=":", label="Identity (no act.)")
ax_fn.set_xlabel("Pre-activation z", fontsize=11)
ax_fn.set_ylabel("Activation output", fontsize=11)
ax_fn.set_title("Activation Functions — ReLU used between MLP layers",
                fontsize=11, fontweight="bold")
ax_fn.legend(fontsize=10)
ax_fn.axhline(0, color="k", lw=0.5)
ax_fn.axvline(0, color="k", lw=0.5)
plt.tight_layout()
plt.show()


### 1.4 NumpyMLP — Full Implementation

We now assemble `linear_layer`, `relu`, and `softmax` into a reusable class.
The forward pass chains these operations through all layers:

$$\mathbf{h}^{(l)} = \mathrm{ReLU}\!\left(\mathbf{h}^{(l-1)} \mathbf{W}^{(l)} + \mathbf{b}^{(l)}\right)$$

Weights are initialised with **Xavier / Glorot** initialisation:

$$W_{ij} \sim \mathcal{N}\!\left(0,\, \frac{2}{n_{\mathrm{in}} + n_{\mathrm{out}}}\right)$$

Training requires backpropagation, which is owned by **05-06**.  Here we focus
exclusively on the forward computation.


In [ ]:
# ── 1.4 NumpyMLP Class ─────────────────────────────────────────────────────────
class NumpyMLP:
    '''Multilayer Perceptron with forward pass implemented in NumPy.

    Weights are initialised with Xavier (Glorot) initialisation.
    Hidden layers use ReLU; the output layer applies softmax.
    Backpropagation is NOT implemented here (see 05-06).

    Attributes:
        layer_dims: List of layer sizes [input, h1, h2, ..., output].
        weights: List of weight matrices W[i] of shape (layer_dims[i], layer_dims[i+1]).
        biases: List of bias vectors b[i] of shape (layer_dims[i+1],).
    '''

    def __init__(self, layer_dims: list[int], seed: int = SEED) -> None:
        '''Initialise NumpyMLP with Xavier weights.

        Args:
            layer_dims: Network architecture, e.g. [784, 256, 128, 10].
            seed: Random seed for reproducible weight initialisation.
        '''
        self.layer_dims = layer_dims
        self.weights: list[np.ndarray] = []
        self.biases:  list[np.ndarray] = []
        rng = np.random.default_rng(seed)
        for i in range(len(layer_dims) - 1):
            n_in, n_out = layer_dims[i], layer_dims[i + 1]
            std = np.sqrt(2.0 / (n_in + n_out))   # Xavier std
            W   = rng.normal(0.0, std, (n_in, n_out))
            b   = np.zeros(n_out)
            self.weights.append(W)
            self.biases.append(b)

    def _relu(self, Z: np.ndarray) -> np.ndarray:
        '''Apply ReLU element-wise.

        Args:
            Z: Pre-activation array of any shape.

        Returns:
            ReLU-activated array of same shape.
        '''
        return np.maximum(0.0, Z)

    def _softmax(self, Z: np.ndarray) -> np.ndarray:
        '''Apply numerically stable softmax over the last axis.

        Args:
            Z: Logit array of shape (n_samples, n_classes).

        Returns:
            Probability array summing to 1 along last axis.
        '''
        Z_s   = Z - Z.max(axis=-1, keepdims=True)
        exp_Z = np.exp(Z_s)
        return exp_Z / exp_Z.sum(axis=-1, keepdims=True)

    def forward(
        self,
        X: np.ndarray,
    ) -> tuple[np.ndarray, list[np.ndarray]]:
        '''Compute the full forward pass through all layers.

        Args:
            X: Input array of shape (n_samples, input_dim).

        Returns:
            Tuple of (output_probs, activations_list).
            output_probs shape: (n_samples, n_classes).
            activations_list[0] = X, activations_list[-1] = output_probs.
        '''
        activations = [X.astype(float)]
        current     = X.astype(float)
        n_layers    = len(self.weights)

        for i, (W, b) in enumerate(zip(self.weights, self.biases)):
            z       = linear_layer(current, W, b)
            is_last = (i == n_layers - 1)
            current = self._softmax(z) if is_last else self._relu(z)
            activations.append(current)

        return current, activations

    def predict(self, X: np.ndarray) -> np.ndarray:
        '''Predict class indices for input samples.

        Args:
            X: Input array of shape (n_samples, input_dim).

        Returns:
            Predicted class indices of shape (n_samples,).
        '''
        probs, _ = self.forward(X)
        return probs.argmax(axis=-1)

    def set_weights_from_torch(self, torch_model: Any) -> None:
        '''Copy weights from a PyTorch nn.Module into this NumpyMLP.

        Expects torch_model.net to be an nn.Sequential containing nn.Linear layers.
        Transposes weight matrices from (n_out, n_in) [PyTorch] to (n_in, n_out) [NumPy].

        Args:
            torch_model: Trained PyTorch MLP whose architecture matches layer_dims.
        '''
        linear_layers = [m for m in torch_model.net if isinstance(m, nn.Linear)]
        assert len(linear_layers) == len(self.weights), "Architecture mismatch"
        for i, layer in enumerate(linear_layers):
            self.weights[i] = layer.weight.detach().cpu().numpy().T  # (n_in, n_out)
            self.biases[i]  = layer.bias.detach().cpu().numpy()

    def count_parameters(self) -> int:
        '''Return the total number of learnable parameters.

        Returns:
            Sum of all weight and bias element counts.
        '''
        total = 0
        for W, b in zip(self.weights, self.biases):
            total += W.size + b.size
        return total

    def __repr__(self) -> str:
        '''Return a string summary of the architecture.'''
        dims_str = " → ".join(str(d) for d in self.layer_dims)
        return f"NumpyMLP({dims_str})  params={self.count_parameters():,}"


# Quick check
mlp_test = NumpyMLP([INPUT_DIM, 256, 128, NUM_CLASSES])
print(mlp_test)
X_dummy  = np.random.default_rng(SEED).standard_normal((8, INPUT_DIM))
out, act = mlp_test.forward(X_dummy)
print(f"  Input:  {X_dummy.shape}")
print(f"  Output: {out.shape}  (probabilities, row sums = {out.sum(axis=1).round(4)})")
print(f"  Layer activations: {[a.shape for a in act]}")


### 1.5 Solving XOR with Hand-Crafted Weights

The single Perceptron (05-01) cannot solve XOR because it is not linearly separable.
A 2-layer MLP *can* — by composing two linear operations with a non-linearity it
creates an internal representation that **makes XOR linearly separable**.

Here we manually set weights that implement the logic:

$$\text{XOR}(x_1, x_2) = (x_1 \text{ OR } x_2) \text{ AND NOT } (x_1 \text{ AND } x_2)$$

The two hidden neurons compute the *OR* and *AND* sub-problems; the output neuron
combines them.


In [ ]:
# ── 1.5 Solving XOR with a 2-Layer MLP ───────────────────────────────────────
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y_xor = np.array([0, 1, 1, 0], dtype=int)   # XOR labels

# Manually crafted weights that solve XOR
# Hidden layer: neuron 0 = OR gate, neuron 1 = AND gate
W1_xor = np.array([[1., 1.],
                    [1., 1.]])    # shape (2, 2): (n_in, n_hidden)
b1_xor = np.array([-0.5, -1.5]) # threshold at 0.5 (OR) and 1.5 (AND)

# Output layer: class-1 score = OR - AND, class-0 score = AND - OR
W2_xor = np.array([[-2.,  3.],
                    [ 6., -6.]])   # shape (2, 2): (n_hidden, n_out)
b2_xor = np.array([ 1., -1.])

mlp_xor = NumpyMLP([2, 2, 2])
mlp_xor.weights = [W1_xor, W2_xor]
mlp_xor.biases  = [b1_xor, b2_xor]

probs_xor, acts_xor = mlp_xor.forward(X_xor)
preds_xor            = probs_xor.argmax(axis=-1)

print("XOR solved by 2-layer MLP with hand-crafted weights")
print("=" * 52)
h1_acts = acts_xor[1]  # hidden layer activations
col_h0 = "h0 (OR)"; col_h1 = "h1 (AND)"
print(f"  {'(x1,x2)':>8s}  {'True':>6s}  {col_h0:>8s}  {col_h1:>8s}  {'Pred':>6s}  {'OK?':>5s}")
print("-" * 52)
for i, (x, y_true) in enumerate(zip(X_xor, y_xor)):
    h0, h1    = h1_acts[i, 0], h1_acts[i, 1]
    pred      = preds_xor[i]
    ok        = "✓" if pred == y_true else "✗"
    print(f"  ({int(x[0])},{int(x[1])})       {y_true:>6d}  {h0:>8.2f}  {h1:>8.2f}  {pred:>6d}  {ok:>5s}")

xor_acc = float(np.mean(preds_xor == y_xor))
print(f"\nXOR accuracy: {xor_acc:.4f}")
assert xor_acc == 1.0, "Hand-crafted MLP should achieve 100% on XOR"
print("Assertion passed: 2-layer MLP perfectly solves XOR!")
print("\nKey insight: the hidden layer transforms the input space so that XOR")
print("becomes linearly separable at the output layer.")


In [ ]:
# ── 1.6 Universal Approximation — Empirical Demonstration ────────────────────
# We train small MLPs to approximate sin(x), |x|, and a step function on [-pi, pi].
# This empirically demonstrates that a single hidden layer can approximate
# arbitrary continuous functions.


def train_numpy_approx(
    X_tr: np.ndarray,
    y_tr: np.ndarray,
    hidden_sizes: list[int],
    n_steps: int = 3000,
    lr: float = 1e-2,
) -> tuple[np.ndarray, list[float]]:
    '''Train a regression MLP from scratch (SGD on MSE) for function approximation.

    Uses simple online gradient descent with NumPy for transparency.
    This is a self-contained minimal training loop (no backprop module needed).

    Args:
        X_tr: Input array of shape (n, 1).
        y_tr: Target array of shape (n,).
        hidden_sizes: List of hidden layer widths.
        n_steps: Number of gradient steps (full-batch).
        lr: Learning rate.

    Returns:
        Tuple of (predicted_values on X_tr, loss_history).
    '''
    rng_uat = np.random.default_rng(SEED + 99)
    dims    = [1] + hidden_sizes + [1]
    weights = []
    biases  = []
    for i in range(len(dims) - 1):
        n_in, n_out = dims[i], dims[i + 1]
        std         = np.sqrt(2.0 / (n_in + n_out))
        weights.append(rng_uat.normal(0, std, (n_in, n_out)))
        biases.append(np.zeros(n_out))

    loss_history: list[float] = []
    X_in = X_tr  # (n, 1)
    y_in = y_tr.reshape(-1, 1)

    for step in range(n_steps):
        # ── Forward ───────────────────────────────────────────────────────────
        activations_uat = [X_in]
        current_uat     = X_in
        for i, (W, b) in enumerate(zip(weights, biases)):
            z_uat   = current_uat @ W + b
            is_last_uat = (i == len(weights) - 1)
            current_uat = z_uat if is_last_uat else np.maximum(0.0, z_uat)
            activations_uat.append(current_uat)

        y_pred = activations_uat[-1]          # (n, 1)
        loss   = float(np.mean((y_pred - y_in) ** 2))
        loss_history.append(loss)

        # ── Backward (MSE + ReLU chain rule) ───────────────────────────────
        delta = 2.0 * (y_pred - y_in) / len(X_in)   # (n, 1)
        for i in range(len(weights) - 1, -1, -1):
            a_prev = activations_uat[i]              # (n, n_in)
            dW     = a_prev.T @ delta                # (n_in, n_out)
            db     = delta.sum(axis=0)               # (n_out,)
            weights[i] -= lr * dW
            biases[i]  -= lr * db
            if i > 0:
                delta = delta @ weights[i].T         # (n, n_in)
                delta = delta * (activations_uat[i] > 0)  # ReLU gradient

    final_forward = activations_uat[-1].ravel()
    return final_forward, loss_history


# Functions to approximate
x_uat     = np.linspace(-np.pi, np.pi, 300)[:, None]  # (300, 1)
targets   = {
    "sin(x)":    np.sin(x_uat.ravel()),
    "|x|":       np.abs(x_uat.ravel()),
    "step(x)":   (x_uat.ravel() >= 0).astype(float),
}

hidden_configs = {
    "2 neurons":  [2],
    "8 neurons":  [8],
    "32 neurons": [32],
}

fig_uat, axes_uat = plt.subplots(len(targets), len(hidden_configs),
                                  figsize=(14, 9), sharey="row")

for row, (fn_name, y_true) in enumerate(targets.items()):
    for col, (cfg_name, hidden_list) in enumerate(hidden_configs.items()):
        ax = axes_uat[row, col]
        preds_uat, losses_uat = train_numpy_approx(
            x_uat, y_true, hidden_list, n_steps=4000, lr=1e-2,
        )
        ax.plot(x_uat.ravel(), y_true,    color="gray", lw=2, label="True")
        ax.plot(x_uat.ravel(), preds_uat, color="#1f77b4", lw=2, ls="--",
                label=f"{cfg_name} (MSE={losses_uat[-1]:.4f})")
        ax.set_title(f"{fn_name}  ←  {cfg_name}", fontsize=9, fontweight="bold")
        ax.legend(fontsize=7)
        if row == len(targets) - 1:
            ax.set_xlabel("x", fontsize=9)
        if col == 0:
            ax.set_ylabel(fn_name, fontsize=9)

plt.suptitle("Universal Approximation: Single Hidden Layer MLPs Fit Arbitrary Functions",
             fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()

print("Observation: more hidden neurons = better approximation.")
print("The UAT guarantees existence of a solution; it doesn't specify how to find it.")


---
## Part 2 — Assembly: MLP as `nn.Module`

We now build a flexible `MLP` class using PyTorch's `nn.Module` pattern.
Key design decisions:

- **Variable depth:** pass `layer_dims = [input_dim, h1, h2, ..., n_classes]`
- **ReLU between layers** (no activation after the final layer — `nn.CrossEntropyLoss` expects raw logits)
- **`nn.Flatten()` built in:** accepts `(N, C, H, W)` images or `(N, d)` flat vectors
- **Verification:** the forward pass must match `NumpyMLP` on the same weights


In [ ]:
# ── 2.1 MLP as nn.Module ──────────────────────────────────────────────────────
class MLP(nn.Module):
    '''Flexible multilayer perceptron as a PyTorch nn.Module.

    Supports arbitrary depth and width. Hidden layers use ReLU; the output
    layer returns raw logits (compatible with nn.CrossEntropyLoss).

    Attributes:
        flatten: nn.Flatten layer to handle image inputs.
        net: nn.Sequential containing alternating Linear and ReLU layers.
    '''

    def __init__(self, layer_dims: list[int]) -> None:
        '''Initialise MLP from a list of layer dimensions.

        Args:
            layer_dims: Architecture specification, e.g. [784, 256, 128, 10].
                        First element = input dim, last = number of output classes.
        '''
        super().__init__()
        self.layer_dims = layer_dims
        self.flatten    = nn.Flatten()

        layers: list[nn.Module] = []
        for i in range(len(layer_dims) - 1):
            layers.append(nn.Linear(layer_dims[i], layer_dims[i + 1]))
            if i < len(layer_dims) - 2:
                layers.append(nn.ReLU())
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        '''Compute forward pass — accepts images or flat vectors.

        Args:
            x: Input tensor of shape (N, C, H, W) for images or (N, d) for flat.

        Returns:
            Logit tensor of shape (N, n_classes).
        '''
        x = self.flatten(x)
        return self.net(x)

    def count_parameters(self) -> int:
        '''Return the total number of trainable parameters.

        Returns:
            Sum of all parameter elements.
        '''
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


# Build and inspect the default MLP
arch_dims = [INPUT_DIM] + HIDDEN_DIMS + [NUM_CLASSES]
model     = MLP(arch_dims).to(device)
print(model)
print(f"\nTotal trainable parameters: {model.count_parameters():,}")

# Show parameter count per layer
for name, param in model.named_parameters():
    print(f"  {name:<30s}  {str(param.shape):<20s}  {param.numel():,}")


In [ ]:
# ── 2.2 Non-Linear Boundary Demo on 2-D Moons ────────────────────────────────
# This cell demonstrates that an MLP can learn CURVED decision boundaries,
# unlike the Perceptron (05-01) which was limited to linear separation.


def plot_nn_boundary(
    model: Any,
    X: np.ndarray,
    y: np.ndarray,
    ax: Any,
    title: str,
    device: torch.device,
    padding: float = 0.4,
) -> None:
    '''Plot the decision boundary of a 2-class nn.Module on 2-D data.

    Args:
        model: PyTorch nn.Module that accepts (N, 2) input and outputs (N, 2) logits.
        X: 2-D feature matrix of shape (n_samples, 2).
        y: Binary labels of shape (n_samples,).
        ax: Matplotlib Axes to draw on.
        title: Plot title.
        device: Device to run inference on.
        padding: Extra margin around the data bounding box.
    '''
    model.eval()
    x_lo = X[:, 0].min() - padding;  x_hi = X[:, 0].max() + padding
    y_lo = X[:, 1].min() - padding;  y_hi = X[:, 1].max() + padding
    xx, yy = np.meshgrid(np.linspace(x_lo, x_hi, 250),
                          np.linspace(y_lo, y_hi, 250))
    grid = torch.tensor(np.column_stack([xx.ravel(), yy.ravel()]),
                         dtype=torch.float32).to(device)
    with torch.no_grad():
        Z = model(grid).argmax(dim=1).cpu().numpy()
    Z = Z.reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.25, cmap=plt.cm.coolwarm)
    ax.contour(xx, yy, Z, levels=[0.5], colors="k", linewidths=1.5)
    colors_2d = np.where(y == 0, "#1f77b4", "#ff7f0e")
    ax.scatter(X[:, 0], X[:, 1], c=colors_2d, s=18, edgecolors="k", lw=0.2, zorder=3)
    ax.set_title(title, fontsize=10, fontweight="bold")
    ax.set_xlabel("x1"); ax.set_ylabel("x2")


# Generate two-moons dataset
X_moon, y_moon = make_moons(n_samples=600, noise=0.2, random_state=SEED)
X_moon_t  = torch.tensor(X_moon, dtype=torch.float32)
y_moon_t  = torch.tensor(y_moon, dtype=torch.long)

# Train a small MLP [2, 16, 8, 2] for 300 steps
mlp_2d     = MLP([2, 16, 8, 2]).to(device)
opt_2d     = torch.optim.Adam(mlp_2d.parameters(), lr=2e-2)
crit_2d    = nn.CrossEntropyLoss()

mlp_2d.train()
for step in range(300):
    opt_2d.zero_grad()
    logits_2d = mlp_2d(X_moon_t.to(device))
    loss_2d   = crit_2d(logits_2d, y_moon_t.to(device))
    loss_2d.backward()
    opt_2d.step()

mlp_2d.eval()
with torch.no_grad():
    preds_2d  = mlp_2d(X_moon_t.to(device)).argmax(dim=1).cpu().numpy()
acc_2d = float(np.mean(preds_2d == y_moon))
print(f"MLP on two-moons after 300 steps: accuracy = {acc_2d:.4f}")

fig_2d, ax_2d = plt.subplots(figsize=(7, 5))
plot_nn_boundary(mlp_2d, X_moon, y_moon, ax_2d,
                 title=f"MLP [2-16-8-2] on Two-Moons (acc={acc_2d:.3f})",
                 device=device)
plt.tight_layout()
plt.show()
print("The MLP learns a CURVED boundary — impossible for a single Perceptron.")


In [ ]:
# ── 2.3 Numerical Equivalence: NumpyMLP vs TorchMLP ─────────────────────────
# Train the nn.Module MLP briefly, then copy weights to NumpyMLP and verify
# that both produce numerically identical forward-pass outputs.

mlp_sanity_torch = MLP(arch_dims).to(device)
opt_sanity       = torch.optim.Adam(mlp_sanity_torch.parameters(), lr=LEARNING_RATE)
crit_sanity      = nn.CrossEntropyLoss()

# 5-batch warm-up on train_loader
mlp_sanity_torch.train()
for i, (imgs, labels) in enumerate(train_loader):
    if i >= 5:
        break
    opt_sanity.zero_grad()
    logits_s = mlp_sanity_torch(imgs.to(device))
    loss_s   = crit_sanity(logits_s, labels.to(device))
    loss_s.backward()
    opt_sanity.step()

# Copy weights to NumpyMLP
mlp_numpy_sanity = NumpyMLP(arch_dims)
mlp_numpy_sanity.set_weights_from_torch(mlp_sanity_torch)

# Generate identical test input
rng_sanity  = np.random.default_rng(SEED + 5)
X_check_np  = rng_sanity.standard_normal((32, INPUT_DIM)).astype(np.float32)
X_check_t   = torch.tensor(X_check_np, dtype=torch.float32).to(device)

# Forward pass with both
mlp_sanity_torch.eval()
with torch.no_grad():
    logits_torch  = mlp_sanity_torch(X_check_t).cpu().numpy()   # (32, 10) logits
    probs_torch   = np.exp(logits_torch - logits_torch.max(axis=1, keepdims=True))
    probs_torch  /= probs_torch.sum(axis=1, keepdims=True)      # softmax for comparison

probs_numpy, _ = mlp_numpy_sanity.forward(X_check_np)           # (32, 10)

# Compare — should be numerically close (< 1e-5 max absolute diff)
max_diff = float(np.abs(probs_numpy - probs_torch).max())
mean_diff = float(np.abs(probs_numpy - probs_torch).mean())
print("Numerical equivalence check: NumpyMLP vs TorchMLP")
print(f"  Max absolute difference:  {max_diff:.2e}")
print(f"  Mean absolute difference: {mean_diff:.2e}")
assert max_diff < 1e-4, f"Forward pass mismatch: max diff = {max_diff:.2e}"
print("Assertion passed: both implementations produce identical outputs.")


---
## Part 3 — Training on FashionMNIST

We train the default MLP architecture `[784, 256, 128, 10]` on FashionMNIST using:

- **Loss:** `nn.CrossEntropyLoss` (multi-class standard)
- **Optimiser:** `Adam` with `lr=1e-3` (see **05-09** for a full optimiser study)
- **Best-model tracking:** save checkpoint at lowest validation loss
- **Target:** > 85 % test accuracy (reasonable for a plain MLP without convolutions)


In [ ]:
# ── Standard Training Functions ───────────────────────────────────────────────
def train_one_epoch(
    model: nn.Module,
    dataloader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    device: torch.device,
) -> tuple[float, float]:
    '''Train for one epoch — multi-class classification.

    Args:
        model: The neural network model.
        dataloader: Training DataLoader.
        optimizer: Optimizer instance.
        criterion: Loss function (CrossEntropyLoss expected).
        device: Device to move tensors to.

    Returns:
        Tuple of (average_loss, accuracy) over the epoch.
    '''
    model.train()
    running_loss = 0.0
    correct      = 0
    total        = 0
    for batch_inputs, batch_targets in dataloader:
        batch_inputs  = batch_inputs.to(device)
        batch_targets = batch_targets.to(device)
        optimizer.zero_grad()
        outputs = model(batch_inputs)
        loss    = criterion(outputs, batch_targets)
        loss.backward()
        optimizer.step()
        running_loss   += loss.item() * batch_inputs.size(0)
        _, predicted    = outputs.max(1)
        correct        += predicted.eq(batch_targets).sum().item()
        total          += batch_targets.size(0)
    return running_loss / total, correct / total


def evaluate(
    model: nn.Module,
    dataloader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
) -> tuple[float, float]:
    '''Evaluate model on a dataset split.

    Args:
        model: The neural network model.
        dataloader: Validation or test DataLoader.
        criterion: Loss function.
        device: Device to move tensors to.

    Returns:
        Tuple of (average_loss, accuracy).
    '''
    model.eval()
    running_loss = 0.0
    correct      = 0
    total        = 0
    with torch.no_grad():
        for batch_inputs, batch_targets in dataloader:
            batch_inputs  = batch_inputs.to(device)
            batch_targets = batch_targets.to(device)
            outputs       = model(batch_inputs)
            loss          = criterion(outputs, batch_targets)
            running_loss += loss.item() * batch_inputs.size(0)
            _, predicted  = outputs.max(1)
            correct      += predicted.eq(batch_targets).sum().item()
            total        += batch_targets.size(0)
    return running_loss / total, correct / total


def plot_training_curves(
    train_losses: list[float],
    val_losses: list[float],
    train_accs: list[float] | None = None,
    val_accs: list[float] | None = None,
) -> None:
    '''Plot loss and optional accuracy curves after training.

    Args:
        train_losses: Training loss per epoch.
        val_losses: Validation loss per epoch.
        train_accs: Optional training accuracy per epoch.
        val_accs: Optional validation accuracy per epoch.
    '''
    ncols = 2 if (train_accs is not None and val_accs is not None) else 1
    fig_tc, axes_tc = plt.subplots(1, ncols, figsize=(12 if ncols == 2 else 6, 4))
    if ncols == 1:
        axes_tc = [axes_tc]
    axes_tc[0].plot(train_losses, label="Train"); axes_tc[0].plot(val_losses, label="Val")
    axes_tc[0].set_xlabel("Epoch"); axes_tc[0].set_ylabel("Loss")
    axes_tc[0].set_title("Loss Curve"); axes_tc[0].legend()
    if ncols == 2 and train_accs and val_accs:
        axes_tc[1].plot(train_accs, label="Train"); axes_tc[1].plot(val_accs, label="Val")
        axes_tc[1].set_xlabel("Epoch"); axes_tc[1].set_ylabel("Accuracy")
        axes_tc[1].set_title("Accuracy Curve"); axes_tc[1].legend()
    plt.tight_layout(); plt.show()


print("train_one_epoch, evaluate, plot_training_curves defined.")


In [ ]:
# ── Train the Default MLP on FashionMNIST ────────────────────────────────────
model     = MLP(arch_dims).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

train_losses: list[float] = []
val_losses:   list[float] = []
train_accs:   list[float] = []
val_accs:     list[float] = []
best_val_loss    = float("inf")
best_model_state = None

for epoch in range(NUM_EPOCHS):
    t0 = time.time()
    train_loss, train_acc = train_one_epoch(
        model, train_loader, optimizer, criterion, device,
    )
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)
    elapsed = time.time() - t0

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    if val_loss < best_val_loss:
        best_val_loss    = val_loss
        best_model_state = {k: v.clone() for k, v in model.state_dict().items()}

    print(f"Epoch {epoch + 1:02d}/{NUM_EPOCHS} | "
          f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
          f"Val Acc: {val_acc:.2%} | Time: {elapsed:.1f}s")

model.load_state_dict(best_model_state)
test_loss, test_acc = evaluate(model, test_loader, criterion, device)
print(f"\nTest accuracy (best model): {test_acc:.4f}")
plot_training_curves(train_losses, val_losses, train_accs, val_accs)


### 3.2 Equivalence Check — NumpyMLP on Test Set

We copy the trained weights into `NumpyMLP` and run the test set through the
NumPy forward pass.  Predictions should match the PyTorch model exactly,
confirming our from-scratch implementation is correct.


### 3.3 sklearn MLPClassifier Cross-Check

We train `sklearn.neural_network.MLPClassifier` with the same architecture (256, 128) on a 10K subset and compare test accuracy.  Both are MLP implementations — the key difference is training data size, not model quality.


In [ ]:
# ── sklearn MLPClassifier Comparison ─────────────────────────────────────────
# We train sklearn's MLPClassifier on a 10K subset (full dataset is slow on CPU)
# and compare its accuracy against our PyTorch MLP.


N_SKLEARN = 10_000   # subset size for sklearn (full 54K is very slow)
rng_sk    = np.random.default_rng(SEED + 11)
sk_idx    = rng_sk.choice(len(X_test_np), size=N_SKLEARN, replace=False)

# Use raw (normalised) images from the training set
all_train_imgs:   list[np.ndarray] = []
all_train_labels: list[int]        = []
model.eval()
with torch.no_grad():
    for imgs, labels in train_loader:
        flat_imgs = imgs.view(imgs.size(0), -1).cpu().numpy()
        all_train_imgs.extend(flat_imgs)
        all_train_labels.extend(labels.numpy())
        if len(all_train_imgs) >= N_SKLEARN:
            break

X_sk_train = np.array(all_train_imgs[:N_SKLEARN])
y_sk_train = np.array(all_train_labels[:N_SKLEARN])
X_sk_test  = X_test_np
y_sk_test  = y_test_np

scaler_sk  = StandardScaler()
X_sk_train = scaler_sk.fit_transform(X_sk_train)
X_sk_test  = scaler_sk.transform(X_sk_test)

t_sk_start = time.time()
sklearn_mlp = MLPClassifier(
    hidden_layer_sizes=(256, 128),
    activation="relu",
    solver="adam",
    learning_rate_init=LEARNING_RATE,
    max_iter=10,
    random_state=SEED,
    verbose=False,
)
sklearn_mlp.fit(X_sk_train, y_sk_train)
t_sk_elapsed = time.time() - t_sk_start

sk_test_acc = sklearn_mlp.score(X_sk_test, y_sk_test)

comparison_rows = [
    {
        "Model":        f"PyTorch MLP (trained on {train_size:,} samples)",
        "Architecture": str(arch_dims),
        "Test Acc":     f"{test_acc:.4f}",
        "Train Time":   f"{sum(t for t in [0]):>6.1f}s",  # already done
        "Framework":    "PyTorch",
    },
    {
        "Model":        f"sklearn MLP (trained on {N_SKLEARN:,} samples)",
        "Architecture": "(256, 128)",
        "Test Acc":     f"{sk_test_acc:.4f}",
        "Train Time":   f"{t_sk_elapsed:>6.1f}s",
        "Framework":    "sklearn",
    },
]
comp_df = pd.DataFrame(comparison_rows)
print("sklearn MLPClassifier vs PyTorch MLP:")
print(comp_df.to_string(index=False))
print(f"\nNote: sklearn trained on only {N_SKLEARN:,} samples vs {train_size:,} for PyTorch.")
print("The accuracy gap is primarily due to data, not the underlying model.")
print("Both implement the same MLP architecture with Adam optimisation.")


In [ ]:
# ── Copy trained weights to NumpyMLP and verify on full test set ──────────────
numpy_mlp_trained = NumpyMLP(arch_dims)
numpy_mlp_trained.set_weights_from_torch(model)

# Collect all test images as a flat NumPy array
all_test_imgs:   list[np.ndarray] = []
all_test_labels: list[int]        = []
all_torch_preds: list[int]        = []

model.eval()
with torch.no_grad():
    for imgs, labels in test_loader:
        logits_cmp = model(imgs.to(device))
        preds_cmp  = logits_cmp.argmax(dim=1).cpu().numpy()
        flat_imgs  = imgs.view(imgs.size(0), -1).cpu().numpy()
        # Undo normalisation for NumpyMLP (it expects [0,1]-ish values)
        # Actually, NumpyMLP will see normalised values — that's fine since
        # we're just verifying numerical equivalence, not input scale
        all_test_imgs.extend(flat_imgs)
        all_test_labels.extend(labels.numpy())
        all_torch_preds.extend(preds_cmp)

X_test_np   = np.array(all_test_imgs)    # (10000, 784)
y_test_np   = np.array(all_test_labels)  # (10000,)
y_torch_np  = np.array(all_torch_preds)  # (10000,)

# Run NumpyMLP forward in batches (full forward at once may be large)
batch_sz   = 1000
numpy_preds: list[int] = []
for start in range(0, len(X_test_np), batch_sz):
    batch = X_test_np[start:start + batch_sz]
    numpy_preds.extend(numpy_mlp_trained.predict(batch).tolist())
numpy_preds_arr = np.array(numpy_preds)

agreement = float(np.mean(numpy_preds_arr == y_torch_np))
numpy_acc  = float(np.mean(numpy_preds_arr == y_test_np))

print("NumpyMLP vs TorchMLP on full test set:")
print(f"  TorchMLP test accuracy:  {test_acc:.4f}")
print(f"  NumpyMLP test accuracy:  {numpy_acc:.4f}")
print(f"  Prediction agreement:    {agreement:.6f}")
assert agreement > 0.999, f"Prediction disagreement: {1 - agreement:.4f}"
print("\nAssertion passed: NumpyMLP and TorchMLP agree on >99.9% of test predictions.")


### 4.0 Interpreting Learned Weights

The first linear layer $\mathbf{W}^{(1)} \in \mathbb{R}^{784 \times 256}$ can be interpreted as 256 **learned templates** (one per output neuron): each row, reshaped to 28×28, shows which pixel patterns activate that neuron most strongly.


In [ ]:
# ── First-Layer Weight Visualisation ─────────────────────────────────────────
# The first linear layer maps 784 → 256.  Each of the 256 output neurons has
# a 784-dim weight vector that can be reshaped to 28×28 and interpreted as a
# "template" or "detector" — it fires most strongly for images that look like it.

first_linear = [m for m in model.net if isinstance(m, nn.Linear)][0]
W_first      = first_linear.weight.detach().cpu().numpy()   # (256, 784)

# Show the first 32 templates as a 4×8 grid
n_show  = 32
ncols_w = 8
nrows_w = n_show // ncols_w

fig_wv, axes_wv = plt.subplots(nrows_w, ncols_w, figsize=(16, 4 * nrows_w))
axes_wv_flat = axes_wv.ravel()

for k in range(n_show):
    template = W_first[k].reshape(28, 28)
    v_max    = np.abs(template).max()
    axes_wv_flat[k].imshow(template, cmap="RdBu_r", vmin=-v_max, vmax=v_max)
    axes_wv_flat[k].set_title(f"N{k}", fontsize=6)
    axes_wv_flat[k].axis("off")

plt.suptitle("First-Layer Weight Templates (784 → 256)  — red=positive, blue=negative",
             fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()

# Distribution of weight magnitudes across all layers
fig_wd, ax_wd = plt.subplots(figsize=(9, 3.5))
layer_linears = [m for m in model.net if isinstance(m, nn.Linear)]
colors_wd     = ["#1f77b4", "#ff7f0e", "#2ca02c"]

for lyr_idx, (lyr, col) in enumerate(zip(layer_linears, colors_wd)):
    w_flat = lyr.weight.detach().cpu().numpy().ravel()
    lbl    = f"Layer {lyr_idx + 1} ({lyr.weight.shape[1]}→{lyr.weight.shape[0]})"
    ax_wd.hist(w_flat, bins=80, alpha=0.5, color=col, label=lbl, density=True)

ax_wd.set_xlabel("Weight value", fontsize=11)
ax_wd.set_ylabel("Density", fontsize=11)
ax_wd.set_title("Weight Value Distributions Across MLP Layers", fontsize=11,
                fontweight="bold")
ax_wd.axvline(0, color="k", lw=1, ls="--")
ax_wd.legend(fontsize=9)
plt.tight_layout()
plt.show()

# Statistics
for lyr_idx, lyr in enumerate(layer_linears):
    w = lyr.weight.detach().cpu().numpy()
    print(f"  Layer {lyr_idx + 1}  shape={w.shape}  "
          f"mean={w.mean():.4f}  std={w.std():.4f}  "
          f"|w|_max={np.abs(w).max():.4f}")


---
## Part 4 — Evaluation & Analysis

### 4.1 Confusion Matrix & Per-Class Accuracy


In [ ]:
# ── 4.1 Confusion Matrix ─────────────────────────────────────────────────────

cm = confusion_matrix(y_test_np, y_torch_np)

fig_cm, ax_cm = plt.subplots(figsize=(9, 7))
im_cm = ax_cm.imshow(cm, cmap="Blues")
plt.colorbar(im_cm, ax=ax_cm)
ax_cm.set_xticks(range(NUM_CLASSES))
ax_cm.set_xticklabels(FASHION_CLASSES, rotation=45, ha="right", fontsize=8)
ax_cm.set_yticks(range(NUM_CLASSES))
ax_cm.set_yticklabels(FASHION_CLASSES, fontsize=8)
ax_cm.set_xlabel("Predicted", fontsize=11)
ax_cm.set_ylabel("True", fontsize=11)
ax_cm.set_title("MLP — FashionMNIST Test Confusion Matrix", fontsize=11,
                fontweight="bold")
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        cell_color = "white" if cm[i, j] > cm[i].max() / 2 else "black"
        ax_cm.text(j, i, str(cm[i, j]), ha="center", va="center",
                   fontsize=7, color=cell_color)
plt.tight_layout()
plt.show()

# Per-class accuracy
print("Per-class accuracy:")
col_cls = "Class"; col_acc = "Accuracy"; col_n = "N test"
print(f"  {col_cls:<14s}  {col_acc:>10s}  {col_n:>8s}")
print("-" * 38)
for i, cls_name in enumerate(FASHION_CLASSES):
    cls_mask  = y_test_np == i
    cls_acc   = float(np.mean(y_torch_np[cls_mask] == i))
    cls_count = int(cls_mask.sum())
    print(f"  {cls_name:<14s}  {cls_acc:>10.4f}  {cls_count:>8d}")

print("\nFull classification report:")
print(classification_report(y_test_np, y_torch_np, target_names=FASHION_CLASSES))


### 4.2b Hidden Layer Representations

We extract activations from the **last hidden layer** (just before the output) and project them to 2-D with PCA.  This reveals how well the MLP has learned to separate the 10 FashionMNIST classes in its internal representation space.


In [ ]:
# ── Hidden Layer Representation: PCA Projection ───────────────────────────────
# Extract the activations at the last hidden layer for the test set,
# project to 2-D with PCA, and plot by class.  This reveals how the MLP
# transforms the 784-D input into an (approximately) linearly separable space.


# Register a hook to capture hidden activations
hidden_activations: list[torch.Tensor] = []


def _hook(module: nn.Module, inp: tuple, out: torch.Tensor) -> None:
    '''Forward hook to capture activations from a specific layer.

    Args:
        module: The layer the hook is registered on.
        inp: Tuple of input tensors to the layer.
        out: Output tensor from the layer.
    '''
    hidden_activations.append(out.detach().cpu())


# Register hook on the ReLU after the second-to-last Linear layer
# (the activation just before the output layer)
penultimate_relu = [m for m in model.net if isinstance(m, nn.ReLU)][-1]
hook_handle      = penultimate_relu.register_forward_hook(_hook)

# Run forward pass on test set (collect activations)
model.eval()
n_pca_samples = 2000   # subset for speed
rng_pca = np.random.default_rng(SEED + 7)
pca_idx = rng_pca.choice(len(X_test_np), size=n_pca_samples, replace=False)
X_pca_subset = X_test_np[pca_idx]
y_pca_subset = y_test_np[pca_idx]

with torch.no_grad():
    for start in range(0, n_pca_samples, BATCH_SIZE):
        batch = torch.tensor(X_pca_subset[start:start + BATCH_SIZE],
                             dtype=torch.float32).to(device)
        model(batch)

hook_handle.remove()
hidden_acts = torch.cat(hidden_activations, dim=0).numpy()   # (n_pca, 128)
hidden_activations.clear()

# PCA → 2D
pca_model   = PCA(n_components=2, random_state=SEED)
hidden_2d   = pca_model.fit_transform(hidden_acts)
var_ratio   = pca_model.explained_variance_ratio_

# Plot by class
fig_pca, ax_pca = plt.subplots(figsize=(9, 7))
cmap_pca = plt.cm.get_cmap("tab10", NUM_CLASSES)

for cls_idx in range(NUM_CLASSES):
    cls_mask = y_pca_subset == cls_idx
    ax_pca.scatter(
        hidden_2d[cls_mask, 0], hidden_2d[cls_mask, 1],
        s=10, alpha=0.6, color=cmap_pca(cls_idx),
        label=FASHION_CLASSES[cls_idx],
    )

ax_pca.set_xlabel(f"PC1 ({var_ratio[0]*100:.1f}% var)", fontsize=11)
ax_pca.set_ylabel(f"PC2 ({var_ratio[1]*100:.1f}% var)", fontsize=11)
ax_pca.set_title("PCA of Last Hidden Layer Activations — Coloured by Class",
                  fontsize=11, fontweight="bold")
ax_pca.legend(fontsize=8, markerscale=3, loc="upper right",
              bbox_to_anchor=(1.18, 1.0))
plt.tight_layout()
plt.show()

print(f"Hidden representation shape: {hidden_acts.shape}  (after last ReLU)")
print(f"PCA variance explained: PC1={var_ratio[0]*100:.1f}%  PC2={var_ratio[1]*100:.1f}%")
print("Classes that cluster well: Trouser, Sneaker are typically well-separated.")
print("Confused classes (e.g. Shirt/T-shirt/Pullover) overlap in the 2-D projection.")


In [ ]:
# ── 4.2 Error Analysis — Hardest Classes ─────────────────────────────────────
# Find the most confused class pairs and visualise misclassified samples

# Top confused pairs (off-diagonal elements)
confused_pairs: list[tuple[int, int, int]] = []
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        if i != j and cm[i, j] > 0:
            confused_pairs.append((cm[i, j], i, j))
confused_pairs.sort(reverse=True)

print("Most confused class pairs:")
col_true = "True"; col_pred = "Predicted"; col_cnt = "Count"
print(f"  {col_true:<14s}  {col_pred:<14s}  {col_cnt:>6s}")
print("-" * 40)
for count, true_cls, pred_cls in confused_pairs[:8]:
    print(f"  {FASHION_CLASSES[true_cls]:<14s}  {FASHION_CLASSES[pred_cls]:<14s}  {count:>6d}")

# Visualise misclassified samples from the top 4 pairs
fig_err, axes_err = plt.subplots(4, 8, figsize=(16, 8))
misclassified_global = np.where(y_torch_np != y_test_np)[0]

for row_idx, (_, true_cls, pred_cls) in enumerate(confused_pairs[:4]):
    pair_idxs = [idx for idx in misclassified_global
                 if y_test_np[idx] == true_cls and y_torch_np[idx] == pred_cls][:8]
    for col_idx in range(8):
        ax = axes_err[row_idx, col_idx]
        if col_idx < len(pair_idxs):
            img = X_test_np[pair_idxs[col_idx]].reshape(28, 28)
            ax.imshow(img, cmap="gray")
            ax.set_title(
                f"T:{FASHION_CLASSES[true_cls][:4]} P:{FASHION_CLASSES[pred_cls][:4]}",
                fontsize=6, color="red",
            )
        ax.axis("off")

plt.suptitle("Misclassified Samples — Top 4 Confused Pairs",
             fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()


### 4.3 Depth vs Width Ablation

A key design choice in MLP architectures: should we use **more layers (depth)** or
**wider layers (width)**?

The Universal Approximation Theorem guarantees that *width alone* is sufficient, but
empirically **depth is more parameter-efficient**:

- Shallow wide networks can memorise but may not generalise well.
- Deeper networks learn hierarchical representations.

We train four architectures of broadly similar parameter counts for `NUM_EPOCHS_ABLAT`
epochs and compare validation accuracy and training speed.


In [ ]:
# ── 4.3 Depth vs Width Ablation on FashionMNIST ──────────────────────────────
ablation_archs = {
    "Shallow-Wide   [784→1024→10]":        [INPUT_DIM, 1024, NUM_CLASSES],
    "2-Layer   [784→256→128→10]":          [INPUT_DIM, 256, 128, NUM_CLASSES],
    "3-Layer   [784→128→128→128→10]":      [INPUT_DIM, 128, 128, 128, NUM_CLASSES],
    "4-Layer   [784→64→64→64→64→10]":      [INPUT_DIM, 64, 64, 64, 64, NUM_CLASSES],
}

ablation_rows: list[dict] = []
fig_ab, ax_ab = plt.subplots(figsize=(9, 5))

for arch_name, dims in ablation_archs.items():
    mlp_ab    = MLP(dims).to(device)
    opt_ab    = torch.optim.Adam(mlp_ab.parameters(), lr=LEARNING_RATE)
    crit_ab   = nn.CrossEntropyLoss()
    val_accs_ab: list[float] = []
    t_start = time.time()

    for ep in range(NUM_EPOCHS_ABLAT):
        train_one_epoch(mlp_ab, train_loader, opt_ab, crit_ab, device)
        _, val_acc_ab = evaluate(mlp_ab, val_loader, crit_ab, device)
        val_accs_ab.append(val_acc_ab)

    elapsed_ab = time.time() - t_start
    final_val  = val_accs_ab[-1]
    n_params   = mlp_ab.count_parameters()

    ablation_rows.append({
        "Architecture": arch_name.strip(),
        "Params":       n_params,
        "Val Acc":      round(final_val, 4),
        "Train Time":   round(elapsed_ab, 1),
        "Depth":        len(dims) - 1,
    })
    ax_ab.plot(
        range(1, NUM_EPOCHS_ABLAT + 1),
        val_accs_ab,
        marker="o", markersize=5, lw=2, label=arch_name.strip(),
    )
    print(f"  {arch_name.strip():<45s}  params={n_params:>8,}  "
          f"val_acc={final_val:.4f}  time={elapsed_ab:.1f}s")

ax_ab.set_xlabel("Epoch", fontsize=11)
ax_ab.set_ylabel("Validation Accuracy", fontsize=11)
ax_ab.set_title(f"Depth vs Width: Val Accuracy over {NUM_EPOCHS_ABLAT} Epochs",
                fontsize=11, fontweight="bold")
ax_ab.legend(fontsize=8, loc="lower right")
plt.tight_layout()
plt.show()

ablation_df = pd.DataFrame(ablation_rows).sort_values("Val Acc", ascending=False)
print("\nAblation Summary (sorted by Val Acc):")
print(ablation_df.to_string(index=False))
print("\nKey observation: moderate depth (2-3 layers) often beats shallow-wide or very deep-narrow")
print("at the same parameter budget and short training duration.")


# ── Parameter efficiency: accuracy per 10K parameters ────────────────────────
print("\nParameter Efficiency (Val Acc per 10K params):")
for row in ablation_rows:
    eff = row["Val Acc"] / (row["Params"] / 10_000)
    arch_lbl = row["Architecture"]
    print(f"  {arch_lbl:<45s}  {eff:.4f} acc per 10K params")

# Bar chart: parameter count per architecture (annotated with val accuracy)
fig_params, ax_params = plt.subplots(figsize=(9, 4))
arch_short   = [r["Architecture"].split("[")[0].strip() for r in ablation_rows]
param_counts = [r["Params"] for r in ablation_rows]
val_finals   = [r["Val Acc"] for r in ablation_rows]
bar_cols_p   = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]
x_pos_p      = list(range(len(ablation_rows)))

bars_p = ax_params.bar(x_pos_p, param_counts, color=bar_cols_p,
                        edgecolor="k", lw=0.8)
for bar_p, acc_p in zip(bars_p, val_finals):
    bar_top = bar_p.get_height()
    ax_params.text(
        bar_p.get_x() + bar_p.get_width() / 2.0,
        bar_top + bar_top * 0.01,
        f"acc={acc_p:.3f}",
        ha="center", va="bottom", fontsize=9,
    )
ax_params.set_xticks(x_pos_p)
ax_params.set_xticklabels(arch_short, fontsize=9)
ax_params.set_ylabel("Total Parameters", fontsize=11)
ax_params.set_title("Parameter Count vs Architecture (annotated with Val Acc)",
                     fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()


---
## Part 5 — Summary & Lessons Learned

### Key Takeaways

1. **Non-linearity is essential.** Without activation functions between layers, stacked
   linear transformations collapse into a single linear transformation — no expressiveness
   is gained.  Adding ReLU breaks this and enables learning curved decision boundaries.

2. **MLPs can solve XOR** (and any function) because the hidden layer creates a new
   internal representation where the problem becomes linearly separable.

3. **Universal Approximation Theorem:** A single hidden layer with sufficient width can
   approximate any continuous function.  Empirically, *depth* (more layers) is more
   parameter-efficient than *width* (fewer, wider layers).

4. **Forward pass = composition of linear + activation operations.**  Training uses
   backpropagation to compute gradients (→ 05-06) and an optimiser to update weights
   (→ 05-09).

5. **NumpyMLP and `nn.Module` MLP are numerically equivalent** at the same weights,
   validating our from-scratch implementation.

### What's Next

→ **05-03 (Activation Functions)** studies ReLU, GELU, Swish, Tanh, and the vanishing
gradient problem in depth.

### Going Further

- Cybenko, G. (1989). *Approximations by superpositions of sigmoidal functions.*
  Mathematics of Control, Signals, and Systems.
- Hornik, K. (1991). *Approximation capabilities of multilayer feedforward networks.*
  Neural Networks.
- Goodfellow, I., Bengio, Y., & Courville, A. (2016). *Deep Learning.* Chapter 6.


In [ ]:
# ── Final Assertions & Summary ────────────────────────────────────────────────
assert test_acc > 0.85,     f"MLP FashionMNIST test accuracy {test_acc:.4f} should be > 85%"
assert xor_acc == 1.0,     "Hand-crafted MLP should achieve 100% on XOR"
assert agreement > 0.999,     f"NumpyMLP vs TorchMLP agreement {agreement:.4f} should be > 99.9%"
assert max_diff < 1e-4,     f"Forward pass max absolute diff {max_diff:.2e} should be < 1e-4"
assert acc_2d > 0.85,     f"MLP on two-moons should exceed 85% accuracy (got {acc_2d:.4f})"

print("All assertions passed.")
print()
print("=" * 62)
print("05-02 — Multilayer Perceptrons: SUMMARY")
print("=" * 62)
print(f"  NumpyMLP XOR (hand-crafted):     {xor_acc:.4f}")
print(f"  MLP two-moons accuracy:          {acc_2d:.4f}")
print(f"  MLP FashionMNIST test accuracy:  {test_acc:.4f}")
print(f"  NumpyMLP / TorchMLP agreement:   {agreement:.6f}")
print(f"  Max forward pass diff (numpy):   {max_diff:.2e}")
print("=" * 62)
print("Next: 05-03 — Activation Functions")
